# Oversampling

### <i> SDV using the `Condition` method.

<i> This is really useful for when you have rare, under-represented cases. Instead of creating thousands of rows, hoping to get that ONE row with the values you want, you can force SDV to create rows that meet this condition.

In [1]:
# Imports
from sdv.single_table import CTGANSynthesizer
from sdv.single_table import GaussianCopulaSynthesizer
from sdv.metadata import Metadata
from sdv.sampling import Condition
import pandas as pd
import seaborn as sns

# Load a new dataframe.
df = pd.read_csv("../../houses.csv")

df.drop(["id", "date", "lat", "long", 'zipcode'], axis=1, inplace=True)
df.shape

(21613, 16)

---

### SDV Condition

In [2]:
# Let's let SDV metadata figure out what each column is containing.
metadata = Metadata.detect_from_dataframe(
    data=df
)

# You can set conditions here to generate more rows for.

#  generate 3000 rows of data
# where waterfront == 1
# you can add other conditions as well
condition = Condition(
    num_rows=3000,
    column_values={
        "waterfront": 1,
        # "view": 4
    }
)

# Set the synthetsizer and fit
synthesizer = GaussianCopulaSynthesizer(metadata)
synthesizer.fit(df)

# Extract synthesized data from those specific conditions
synthetic_data = synthesizer.sample_from_conditions(
    conditions=[condition]
)

# Output dataframe
synthetic_data

c:\Users\tuomas.valtanen\dataeng_advda2026_lectures\DataEngineering_AdvDA_2026\.venv\Lib\site-packages\sdv\single_table\base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Sampling conditions: 100%|██████████| 3000/3000 [00:00<00:00, 11846.26it/s]


,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
0,955568.0,4,3.73,2410,6152,2.5,1,0,3,9,1904,82,1999,2006,2269,7286
1,454093.0,3,2.42,2002,16315,2.1,1,0,4,6,1141,524,1960,2006,2138,16399
2,204605.0,4,1.27,1571,4859,1.6,1,0,3,6,1142,304,1945,1994,1540,1892
3,755487.0,3,2.15,1740,26238,1.6,1,0,3,8,1832,2,1952,2008,2119,13209
4,340688.0,3,2.13,1560,1636,2.4,1,0,3,7,2128,0,2005,1876,1519,1386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2995,382031.0,3,1.60,2199,16028,1.8,1,0,3,8,2099,2,1962,2005,2131,24587
2996,356195.0,3,1.30,603,4239,1.5,1,0,3,5,535,625,1926,2006,1050,2566
2997,431323.0,2,1.39,1508,63769,1.5,1,0,4,7,977,723,1933,1988,1566,46007
2998,679634.0,2,2.20,1951,15376,2.2,1,0,3,8,2033,20,1985,1975,1648,24721


---

### Concatenating the original with the synthetic data

In [3]:
# And this is how we combine the synthetic with the real dataset.
df_combined = pd.concat([df, synthetic_data])

In [4]:
df['waterfront'].value_counts()

waterfront
0    21450
1      163
Name: count, dtype: int64

In [5]:
df_combined['waterfront'].value_counts()

waterfront
0    21450
1     3163
Name: count, dtype: int64

In [6]:
# switching to decimal notation for the variance checks
pd.set_option('display.float_format', lambda x: "%.2f" % x)

In [7]:
df.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00,21613.00
mean,540088.14,3.37,2.11,2079.90,15106.97,1.49,0.01,0.23,3.41,7.66,1788.39,291.51,1971.01,84.40,1986.55,12768.46
std,367127.20,0.93,0.77,918.44,41420.51,0.54,0.09,0.77,0.65,1.18,828.09,442.58,29.37,401.68,685.39,27304.18
min,75000.00,0.00,0.00,290.00,520.00,1.00,0.00,0.00,1.00,1.00,290.00,0.00,1900.00,0.00,399.00,651.00
25%,321950.00,3.00,1.75,1427.00,5040.00,1.00,0.00,0.00,3.00,7.00,1190.00,0.00,1951.00,0.00,1490.00,5100.00
50%,450000.00,3.00,2.25,1910.00,7618.00,1.50,0.00,0.00,3.00,7.00,1560.00,0.00,1975.00,0.00,1840.00,7620.00
75%,645000.00,4.00,2.50,2550.00,10688.00,2.00,0.00,0.00,4.00,8.00,2210.00,560.00,1997.00,0.00,2360.00,10083.00
max,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,4820.00,2015.00,2015.00,6210.00,871200.00


In [8]:
df_combined.describe()

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,sqft_living15,sqft_lot15
count,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00,24613.00
mean,543665.10,3.37,2.12,2085.32,15194.92,1.57,0.13,0.24,3.41,7.66,1793.40,270.77,1970.92,316.53,1992.26,12866.18
std,360803.93,0.93,0.77,946.08,39234.99,0.56,0.33,0.77,0.65,1.19,825.08,430.00,29.43,728.00,686.48,25976.52
min,75000.00,0.00,0.00,290.00,520.00,1.00,0.00,0.00,1.00,1.00,290.00,0.00,1900.00,0.00,399.00,651.00
25%,324888.00,3.00,1.59,1410.00,5000.00,1.00,0.00,0.00,3.00,7.00,1200.00,0.00,1951.00,0.00,1490.00,5060.00
50%,457000.00,3.00,2.25,1920.00,7713.00,1.50,0.00,0.00,3.00,7.00,1580.00,0.00,1975.00,0.00,1850.00,7695.00
75%,651667.00,4.00,2.50,2579.00,11535.00,2.00,0.00,0.00,4.00,8.00,2230.00,500.00,1997.00,0.00,2370.00,10625.00
max,7700000.00,33.00,8.00,13540.00,1651359.00,3.50,1.00,4.00,5.00,13.00,9410.00,4820.00,2015.00,2015.00,6210.00,871200.00


In [9]:
# we have now more waterfront houses, but always remember: is this now realistic?
# careful with the amount of synthesized data, always!